# Ledger: General 

This notebook shows the general ledger for the LLC




In [1]:
# Load bookkeeping services
import os
from ledger.LLC import LLC
from pathlib import Path
import datetime
from IPython.display import display, Markdown

top = os.path.join(Path.home(), 'GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group')
llc = LLC('WBGroupLLC',debug=False, top=top)

# 
dtReport = datetime.datetime.now().strftime('%Y.%m.%d')

display(Markdown(f"### Profile - {dtReport}"))
display(Markdown(f"- **LLC Name: {llc.objName}**"))
display(Markdown(f"- **Year: {llc.yr}**"))



### Profile - 2026.03.13

- **LLC Name: WBGroupLLC**

- **Year: 2025**

## General Ledger: Incomes/Expenses

In [3]:
# General Ledger - All accounts
llc._Bank()
glDF = llc.bk.df.groupby(['Acct']).amt.sum()
glDF.loc[f'Balance'] = glDF.sum()
print(glDF.to_string())

Acct
Acct.Asset.Purchase    -213936.95
Acct.Cash.Expense        -1766.92
Acct.Cash.Income          4000.53
Acct.Cash.Investment    219050.00
Acct.Cash.Misc            -106.33
Acct.Cash.Util            -921.15
Acct.Interest.Income       400.00
Balance                   6719.18


## Investment Ledger
- shows investments (cash/property/others) in year by Owner/Manager

In [4]:
# Investment Ledger
import pandas as pd
iDF = llc.bk.df[llc.bk.df.Acct.str.contains('Investment')].copy()
iDF = pd.concat([iDF, llc.bk.df[llc.bk.df.Acct.str.contains('Purchase')].copy()])
iDF = pd.DataFrame(iDF.groupby(['AcctSub','Acct']).amt.sum())

#iDF.columns = ['OwnerID', 'amt']
# Add column
oList = llc.owners()
#iDF['Owner'] = [[d['nm'][0] for d in oList if d['oID'] == ndx][0] for (ndx,acct) in iDF.index][0]
ownList = []
for (ndx,acct) in iDF.index:
    oID = 'LLC'
    for d in oList:
        if d['oID'] == ndx:
            oID = d['nm'][0]
            break
    ownList.append(oID)
iDF['Owner'] = ownList     
iDF.reset_index(level='AcctSub', inplace=True)
# Add row balance
bal = ('Balance_Acct.Cash')
iDF.loc[bal] = [llc.objName, float(iDF.amt.sum()),'LLC']
llcBalInvestment = iDF.amt.iloc[-1]
iDF

,AcctSub,amt,Owner
Acct,,,
Acct.Asset.Purchase,WBGroupLLC,-213936.95,LLC
Acct.Cash.Investment,o20250801-1,219050.00,Francis X Rojas
Balance_Acct.Cash,WBGroupLLC,5113.05,LLC


# Ledger - Income

In [5]:
# Investment Ledger
incDF = llc.bk.df[llc.bk.df.Acct.str.contains('.Income')].copy()
incDF = pd.DataFrame(incDF.groupby(['Acct','AcctSub','dt']).amt.sum())

# --- lookup customer name based on ID
cList = llc.customers()
custList = []
for (acct, ndx,dt) in incDF.index:
    for d in cList:
        if d['oID'] == ndx: custList.append(d['nm'])
        else: custList.append('na')
incDF['Customer'] = custList
incDF.loc[('Total','','')] = [incDF.amt.sum(),'']
incDF

amt  \
Acct                 AcctSub     dt                    
Acct.Cash.Income     c20251001-1 10/23/2025  1000.00   
                                 10/24/2025     0.53   
                                 11/12/2025  1500.00   
                                 12/01/2025  1500.00   
Acct.Interest.Income Bank        10/24/2025   400.00   
Total                                        4400.53   

                                                                         Customer  
Acct                 AcctSub     dt                                                
Acct.Cash.Income     c20251001-1 10/23/2025  [Nicola Rojas, Alejandro Villarreal]  
                                 10/24/2025  [Nicola Rojas, Alejandro Villarreal]  
                                 11/12/2025  [Nicola Rojas, Alejandro Villarreal]  
                                 12/01/2025  [Nicola Rojas, Alejandro Villarreal]  
Acct.Interest.Income Bank        10/24/2025                                    na  
Total

## Ledger - Profit/Loss

In [6]:
# Profit/Loss reconcile with Bank Stmt Balance
df = llc.bk.df.copy()
def _isProfitLoss(self, r):
    if 'Purchase' in r.Acct : return False
    if 'Investment' in r.Acct : return False
    if r.TransType == 'Exp' : return 'Expense'
    return 'Revenue'
df['PnL'] = df.apply(lambda r: _isProfitLoss(llc, r), axis=1)
plDF = pd.DataFrame(df[df.PnL != False])

pvt = pd.pivot_table(plDF, values='amt', index=['PnL', 'Acct'], aggfunc='sum')

# 2. Create subtotals for the top level ('Division')
st = pvt.groupby(level=0).sum()

# 3. Rename the subtotal index to match the MultiIndex structure
#    The second level gets a descriptive name (e.g., 'Division Total')
st.index = pd.MultiIndex.from_product([st.index, ['SubTotal']])

# 4. Combine the original pivot table and the subtotals, then sort
pvtST = pd.concat([pvt, st]).sort_index()

# Optional: Add a grand total using margins on the new table
#pvt_with_subtotals.loc[('Profit-Loss', ''), 'All'] = pvt_with_subtotals.sum().values[0]

#pd.MultiIndex.from_arrays([[subtotals.sum()]],names = ('Profit_Loss','Total'))
pvtST.index
#llcBalInvestmen
#
pNdx = [['Profit_Loss'], ['Total']]
iNdx = [['Investment'], ['Total']]
bkNdx = [['Reconcile'], ['Bank Balance']]


pd.concat([pvtST,
           pd.DataFrame([st.sum()], index = pd.MultiIndex.from_arrays(pNdx, names=('PnL', None))),
           pd.DataFrame([llcBalInvestment], index = pd.MultiIndex.from_arrays(iNdx, names=('PnL', None)), columns=['amt']),
           pd.DataFrame([llcBalInvestment+st.sum().amt], index = pd.MultiIndex.from_arrays(bkNdx, names=('PnL', None)), columns=['amt']),
          ]
         )
#st.sum(), llcBalInvestment
#pd.DataFrame([llcBalInvestment], index = pd.MultiIndex.from_arrays(iNdx, names=('PnL', None)))

amt
PnL                                      
Expense     Acct.Cash.Expense    -1808.02
            Acct.Cash.Misc        -313.33
            Acct.Cash.Util        -921.15
            SubTotal             -3042.50
Revenue     Acct.Cash.Expense       41.10
            Acct.Cash.Income      4000.53
            Acct.Cash.Misc         207.00
            Acct.Interest.Income   400.00
            SubTotal              4648.63
Profit_Loss Total                 1606.13
Investment  Total                 5113.05
Reconcile   Bank Balance          6719.18

## Ledger - Expenses Details

In [7]:
# Investment Ledger
expDF = llc.bk.df[llc.bk.df.Acct.str.contains('Expense')].copy()
expDF[['dt', 'amt', 'AcctSub', 'TDesc']]

,dt,amt,AcctSub,TDesc
7,11/17/2025,-2.00,hays,Expense: 11/15 hays co tx wimber fort worth tx...
8,11/17/2025,-30.00,hays,Expense: 11/15 hays co tx wimber san marcos tx...
9,11/17/2025,-19.47,amazon,Expense: 11/15 amazon mktpl*b80w8 amzn.com/bil...
10,11/17/2025,-32.42,sp,Expense: 11/14 sp growers solutio growerssolut...
13,11/07/2025,-42.15,sp,Expense: 11/06 sp growers solutio growerssolut...
14,11/03/2025,-73.56,amazon,Expense: 11/02 amazon mktpl*nk5ps amzn.com/bil...
15,11/03/2025,-487.13,laird,Expense: 10/31 laird plastics san 469-299-7029...
21,10/22/2025,-135.00,Maintenance,"Repair Utility Outlet,Electrician"
22,10/22/2025,-36.09,wimberley,Expense: 10/22 wimberley ace wimberley tx p000...
24,10/17/2025,-57.31,sq,Expense: 10/16 sq *rodco steel di new braunfel...


## Ledger - Misc Transaction Summary

In [8]:
# Miscellenous / Unclassified Transactions
llc.bk.df[llc.bk.df.Acct.str.contains('Misc')][['dt', 'amt', 'AcctSub', 'desc']]


,dt,amt,AcctSub,desc
0,12/29/2025,-177.00,Misc,Cash eWithdrawal in Branch 12/29/2025 13:47 PM...
1,12/29/2025,177.00,Misc,eDeposit in Branch 12/29/25 03:48:15 PM 14650 ...
2,12/26/2025,-135.80,Misc,ALLSTATE IND CO INS PYMT DEC024 00000043853221...
16,10/24/2025,-0.53,Misc,TRUIST ACCTVERIFY 251024 15280212831 ALEJANDRO...
45,09/29/2025,30.00,Misc,MOBILE DEPOSIT : REF NUMBER :409290718471


## Owner Details

In [9]:
# Owners DB
pd.DataFrame(llc.owners())

,oID,nm,addr,status,pct,kw
0,o20250801-1,[Francis X Rojas],"177 Kingsway Dr, Wimberley, 78676",Manager,0.96,[WT FED#02M03 NATIONAL FINANCIAL]
1,o20250801-2,[Alexandra Rojas],TBD,non_active member,0.02,[]
2,o20250801-3,[Nicola Rojas],"805 High Mesa Dr, Wimberley, 78676",non-active member,0.02,[]


## Customer Details

In [14]:
# Customer DB
pd.DataFrame(llc.customers())

,oID,nm,addr,rent,start,kwList
0,c20251001-1,"[Nicola Rojas, Alejandro Villarreal]","805 High Mesa Dr, Wimberley, 78676",1500.0,2025.10.01,[]


## Asset Details

In [15]:
# Assets DB
pd.DataFrame(llc.assets())

,oID,addr,purchaseValue,purchaseDate,stakeholderPct,kwList
0,p20250826-805HMD,"805 High Mesa Dr, Wimberley, 78676",213936.95,2025.08.26,{'o20250801_1': 100.0},[]


# Accounting-Bookkeeping 101

## accounting ledgers: 

| Ledger | Description |
| :---- | :----: |
|General Ledger | master document; record all transactions; includes all accounts related to a company's assets, liabilities, equity, revenue, and expenses.|
|Sales Ledger | |
|Purchase Ledger |  

## Key Principles (pyApps)

- utilizing pythn accounting libraries
- double-entry bookkeeping, or by
- building a custom application (general ledger for rental LLC)
- a web framework / command-line tool
- read bank statements (often from CSV files)
- perform transactions and reporting. 

### Beancount: 
A Python package for double-entry accounting. You can use its command-line tools to manage financial transactions written in a plain-text file, making it suitable for tracking an LLC's finances and maintaining a Git-based audit trail.
### Blnk Finance: 
An open-source, developer-focused toolkit that includes a double-entry ledger for managing balances and transactions. It offers features like balance monitoring, reconciliation, and identity management, and is designed to help you build fintech products.
### Django Ledger: 
An open-source accounting and financial ledger system built on the Django framework. It's a good option if you need to integrate robust accounting features directly into a Python web application. 

### Github Py Ledgers

1. [**>Confidential Ledger Azure**](https://learn.microsoft.com/en-us/python/api/overview/azure/confidentialledger-readme?view=azure-python#key-concepts)
    - good reference
1. [**> Ledger Visualization**](https://wilw.dev/blog/2022/04/24/ledger-python-visualisation/) : pure text (csv) files, uses vis.py 

## Ledger for Taxes

- bare minimum records
- accounting ledger for an LLC to file taxes
- clear, accurate summary
  - all business income (gross receipts)
  - expenses.
- no specific bookkeeping method
- method used must clearly reflect your income and expenses. 

## key information required for each transaction includes:

- Amount of the transaction.
- Date of the transaction.
- Description of the item purchased or service received.
- Business purpose (why the expense was necessary).
- Source of income or the payee for expenses. 
- Essential Records and Documentation

## Documentation
The IRS requires that you maintain documentation to support the figures reported on your tax return. 

- Well-organized,
- detailed records make tax preparation easier
- help you maximize deductions
- ensuring you have the necessary documentation in case of an IRS audit.
- Records: keep at least three years from the date you filed the return. 


|Record Type 	|Description	|Supporting Documents to Keep|
| ---- | ---- | ---- |
|Income	|All gross receipts from business operations.	|Invoices sent to clients, cash register tapes, deposit slips, bank account deposits, and Forms 1099-NEC received.
|Expenses	|All costs incurred to carry on your business.|	Canceled checks, credit card statements, invoices for purchases, and receipts.|
|Assets & Depreciation|	Records for property like equipment or vehicles used in the business that last more than a year.	|Date and method of acquisition, purchase price, cost of improvements, deductions taken for depreciation, business use, and details of disposition/sale.|
|Payroll (if applicable)|	Records related to employees (not owners, typically).|	Annual W-2s, quarterly and annual payroll tax returns, and complete pay records/time sheets for each employee.|

### Multi-member LLC: Filing Requirements by LLC Structure
- Your LLC's tax classification determines which forms you will need to file, which impacts how you report your income and expenses. 
- Multi-member LLC: The default is to be taxed as a partnership. The LLC files an informational Form 1065 and provides a Schedule K-1 to each member, who then reports their share of profit or loss on their personal Form 1040 using Schedule E.
- LLC taxed as a Corporation (S corp or C corp): If you elect this status, you will file Form 1120 (C corp) or Form 1120-S (S corp). 


# Accounting: Best Practices

## Rental Accounting

|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Acct.Cash.Income     | ##.## |       | yy.mm.dd | CustID  | record the cash received |
|Acct.Rent.Receivable | ##.## |       | yy.mm.dd | CustID  | record amt owed by the tenant|
|Income.Rent.Revenue  |       | ##.## | yy.mm.dd | CustID  | record total income earned, period |

## Initial investment of cash
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Acct.Cash.Investment |       | ##.## | yy.mm.dd |           | record cash investment|
|LLC.Equity.Member    | ##.## |       | yy.mm.dd | memberID  | LLC equity, member % |

## Company purchasing an investment
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|LLC.Prop.Investment  | ##.## |        | yy.mm.dd | propID | record value of purchase/investment|
|Acct.Cash.Purchase   |       | ##.##  | yy.mm.dd | propID | Cash account reduced |

## Expense: Paid by Cash / Debit Card
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Acct.Cash.Expense    |       | ##.##  | yy.mm.dd | expCls | record expense details|
|Acct.Expense.<expCls>| ##.## |        | yy.mm.dd | expCls | record expense details|


## Expense: Paid by accured (credit card/loan-agreement/late payment)
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Liab.Payable.Expense |       | ##.## | yy.mm.dd | expCls | record expense details |
|Liab.Expense.<expCls>| ##.## |       | yy.mm.dd | expCls | record expense details|
| | | | | | future |
|Acct.Cash.Expense    |       | ##.## | yy.mm.dd | expCls | record expense details|
|Acct.Expense.<expCls>| ##.## |       | yy.mm.dd | expCls | record expense details|


Example: Recording rent expense Transaction: A company pays \(\$1,000\) for monthly rent.Debit: The Rent Expense account is debited for \(\$1,000\).Credit: The Cash account is credited for \(\$1,000\). 

Note 3Example: Recording an accrued expense Transaction: At the end of the month, a company has an accrued electricity expense of \(\$300\) that has not yet been paid.Debit: The Electricity Expense account is debited for \(\$300\).Credit: The Accrued Expenses (or Electricity Expense Payable) account is credited for \(\$300\) to create the liability. 


## NOTE 1: Subsequent changes to the investment
- accounting for an investment becomes more complex, 
- value of investment fair value method or equity method,
- depending on the type of investment.

## NOTE 2: Earnings from investments
- recognized in the `income statement`
- dividends received are often treated as either operating or investment inflows
- `cash flow statement`.

## NOTE 3: accrual method
- expenses are recorded when they are incurred, not when they are paid.
- Example: Recording an accrued expense Transaction:
    - end of the month, not paid elect expense 
    - a company has an accrued electricity expense of $300
    - Debited $300 :  Expense.Utility.Elec
    - Credit $300 : Liab.Expense.Utility
    - The Accrued Expenses (or Electricity Expense Payable) - create the liability. 